In [0]:
# Create the landing folder inside your new volume
dbutils.fs.mkdirs("/Volumes/workspace/dev_bronze_layer/raw_files/landing")

In [0]:
# Filename: src/notebooks/bronze_ingestion.py
from pyspark.sql.functions import current_timestamp, col

# Define volume path
source_path = "/Volumes/workspace/dev_bronze_layer/raw_files/landing/"

# Read raw CSV with the _metadata column enabled
raw_df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(source_path)
          # Unity Catalog way to get file path
          .select("*", "_metadata.file_path") 
          .withColumn("ingestion_timestamp", current_timestamp()))

# Rename for audit requirement
bronze_df = raw_df.withColumnRenamed("file_path", "source_file")

# Write to Delta Bronze Layer
bronze_df.write.format("delta").mode("append").saveAsTable("workspace.dev_bronze_layer.events_raw")